In [ ]:
import warnings, os, random
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # headless — swap to TkAgg/inline for notebooks
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (LSTM, Dense, Dropout, BatchNormalization,
                                      Input, Bidirectional)
from tensorflow.keras.callbacks import (EarlyStopping, ReduceLROnPlateau,
                                         ModelCheckpoint)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
import tensorflow.keras.backend as K

from sklearn.preprocessing   import MinMaxScaler, StandardScaler
from sklearn.metrics         import (mean_squared_error, mean_absolute_error,
                                     r2_score, mean_absolute_percentage_error)
from sklearn.model_selection import TimeSeriesSplit

#Reproducibility
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

#Output directory
OUT = "/content/"
os.makedirs(OUT, exist_ok=True)

#Publication-quality matplotlib defaults
COLORS = {
    "lstm"      : "#8E44AD",
    "actual"    : "#1A1A2E",
    "train_bg"  : "#F4ECF7",
    "pred_bg"   : "#FEF9E7",
    "grid"      : "#DDDDDD",
    "pos"       : "#27AE60",
    "neg"       : "#E74C3C",
}
plt.rcParams.update({
    "font.family"       : "DejaVu Serif",
    "font.size"         : 11,
    "axes.titlesize"    : 13,
    "axes.labelsize"    : 12,
    "axes.linewidth"    : 1.2,
    "axes.grid"         : True,
    "grid.color"        : COLORS["grid"],
    "grid.linewidth"    : 0.6,
    "grid.linestyle"    : "--",
    "legend.framealpha" : 0.9,
    "legend.fontsize"   : 10,
    "xtick.labelsize"   : 10,
    "ytick.labelsize"   : 10,
    "figure.dpi"        : 150,
    "savefig.dpi"       : 300,
    "savefig.bbox"      : "tight",
    "savefig.facecolor" : "white",
})

MONTH_NAMES = ["Jan","Feb","Mar","Apr","May","Jun",
               "Jul","Aug","Sep","Oct","Nov","Dec"]

print("="*70)
print("  SHALLOW LSTM — DENGUE PREDICTION, BANGLADESH (2008–2025)")
print("  TensorFlow:", tf.__version__)
print("="*70)

# 1.  DATA LOADING & FEATURE ENGINEERING
df = pd.read_csv("/content/Dengue_Climate_Bangladesh.csv")
df["DATE"]       = pd.to_datetime(df[["YEAR","MONTH"]].assign(DAY=1))
df               = df.sort_values("DATE").reset_index(drop=True)
df["TIME_IDX"]   = np.arange(len(df))

# Engineered features
df["TEMP_RANGE"] = df["MAX"] - df["MIN"]
df["TEMP_MEAN"]  = (df["MAX"] + df["MIN"]) / 2
df["HEAT_INDEX"] = (df["TEMP_MEAN"] * df["HUMIDITY"]) / 100
df["RAIN_LOG"]   = np.log1p(df["RAINFALL"])
df["SIN_MONTH"]  = np.sin(2 * np.pi * df["MONTH"] / 12)
df["COS_MONTH"]  = np.cos(2 * np.pi * df["MONTH"] / 12)
df["DENGUE_LOG"] = np.log1p(df["DENGUE"])

# Lag features
for lag in [1, 2, 3]:
    df[f"DENGUE_LAG{lag}"] = df["DENGUE_LOG"].shift(lag)
    df[f"RAIN_LAG{lag}"]   = df["RAIN_LOG"].shift(lag)
    df[f"HUMID_LAG{lag}"]  = df["HUMIDITY"].shift(lag)

df = df.dropna().reset_index(drop=True)

FEATURES = [
    "TEMP_MEAN","TEMP_RANGE","HEAT_INDEX",
    "HUMIDITY","RAIN_LOG","RAINFALL",
    "SIN_MONTH","COS_MONTH","TIME_IDX",
    "DENGUE_LAG1","DENGUE_LAG2","DENGUE_LAG3",
    "RAIN_LAG1","RAIN_LAG2","HUMID_LAG1",
]
TARGET      = "DENGUE_LOG"
N_FEATURES  = len(FEATURES)
LOOKBACK    = 12   # 12-month sliding window (1 full seasonal cycle)

print(f"\nObservations   : {len(df)}")
print(f"Features       : {N_FEATURES}")
print(f"STM lookback  : {LOOKBACK} months")

# 2.  TRAIN / TEST SPLIT  (≤2024 → train | 2025 → test)
train_mask = df["YEAR"] <= 2024
test_mask  = df["YEAR"] == 2025

df_train = df[train_mask].copy()
df_test  = df[test_mask].copy()

print(f"Training       : {len(df_train)} months (2008–2024)")
print(f"Test (forecast): {len(df_test)} months (2025 Jan–Oct)")

# 3.  SCALING
feat_scaler   = MinMaxScaler(feature_range=(0, 1))
target_scaler = MinMaxScaler(feature_range=(0, 1))

# Fit only on training data
X_train_raw = df_train[FEATURES].values
y_train_raw = df_train[TARGET].values.reshape(-1, 1)

X_train_sc  = feat_scaler.fit_transform(X_train_raw)
y_train_sc  = target_scaler.fit_transform(y_train_raw).ravel()

# Transform test using training scalers
X_test_raw  = df_test[FEATURES].values
y_test_raw  = df_test[TARGET].values.reshape(-1, 1)

X_test_sc   = feat_scaler.transform(X_test_raw)
y_test_sc   = target_scaler.transform(y_test_raw).ravel()

# 4.  SEQUENCE BUILDER  (sliding window → 3-D LSTM input)
def build_sequences(X_sc, y_sc, lookback):
    """
    Returns X of shape (samples, lookback, n_features)
             y of shape (samples,)
    """
    Xs, ys = [], []
    for i in range(lookback, len(X_sc)):
        Xs.append(X_sc[i - lookback : i])
        ys.append(y_sc[i])
    return np.array(Xs), np.array(ys)

# Combine scaled train + test for seamless sequence building
X_all_sc = np.vstack([X_train_sc, X_test_sc])
y_all_sc  = np.concatenate([y_train_sc, y_test_sc])

X_seq, y_seq = build_sequences(X_all_sc, y_all_sc, LOOKBACK)

# Split point: last len(df_test) sequences belong to test
n_test   = len(df_test)
n_train  = len(X_seq) - n_test

X_tr, y_tr = X_seq[:n_train], y_seq[:n_train]
X_te, y_te = X_seq[n_train:], y_seq[n_train:]

print(f"\n✔  LSTM Train     : X{X_tr.shape}  y{y_tr.shape}")
print(f"✔  LSTM Test      : X{X_te.shape}  y{y_te.shape}")

# 5.  SHALLOW LSTM ARCHITECTURE
def build_shallow_lstm(lookback, n_features,
                        units=64, dropout=0.2, lr=1e-3, l2_reg=1e-4):
    """
    Shallow LSTM (single recurrent layer).

    Architecture
    ─────────────
    Input   → (lookback, n_features)
    LSTM    → units neurons, return_sequences=False, L2 kernel regularization
    Dropout → dropout rate
    BN      → BatchNormalization (stabilizes training)
    Dense(32)→ ReLU
    Dense(1) → linear (regression output)
    """
    inp = Input(shape=(lookback, n_features), name="input")

    x = LSTM(units,
             activation="tanh",
             recurrent_activation="sigmoid",
             kernel_regularizer=l2(l2_reg),
             recurrent_regularizer=l2(l2_reg),
             return_sequences=False,
             name="shallow_lstm")(inp)

    x = Dropout(dropout, name="dropout")(x)
    x = BatchNormalization(name="batch_norm")(x)
    x = Dense(32, activation="relu", kernel_regularizer=l2(l2_reg),
              name="dense_hidden")(x)
    out = Dense(1, activation="linear", name="output")(x)

    model = Model(inputs=inp, outputs=out, name="Shallow_LSTM")
    model.compile(optimizer=Adam(learning_rate=lr, clipnorm=1.0),
                  loss="huber",           # robust to outliers
                  metrics=["mae"])
    return model

# 6.  HYPERPARAMETER SEARCH via TimeSeriesSplit Walk-Forward Validation
print("\n─"*35)
print("  HYPERPARAMETER SEARCH (Walk-Forward CV)")
print("─"*35)

param_grid = [
    {"units": 32,  "dropout": 0.1, "lr": 1e-3, "l2_reg": 1e-4},
    {"units": 64,  "dropout": 0.2, "lr": 1e-3, "l2_reg": 1e-4},
    {"units": 64,  "dropout": 0.3, "lr": 5e-4, "l2_reg": 1e-4},
    {"units": 128, "dropout": 0.2, "lr": 1e-3, "l2_reg": 1e-5},
    {"units": 128, "dropout": 0.3, "lr": 5e-4, "l2_reg": 1e-4},
]

tscv     = TimeSeriesSplit(n_splits=5)
best_params  = None
best_val_rmse = np.inf
search_results = []

for params in param_grid:
    fold_rmses = []
    for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_tr)):
        Xf_tr, yf_tr = X_tr[tr_idx], y_tr[tr_idx]
        Xf_val, yf_val = X_tr[val_idx], y_tr[val_idx]

        m = build_shallow_lstm(LOOKBACK, N_FEATURES, **params)
        m.fit(Xf_tr, yf_tr,
              epochs=200, batch_size=16,
              validation_data=(Xf_val, yf_val),
              callbacks=[EarlyStopping(patience=20, restore_best_weights=True,
                                       monitor="val_loss")],
              verbose=0)

        pred_sc  = m.predict(Xf_val, verbose=0).ravel()
        pred_log = target_scaler.inverse_transform(pred_sc.reshape(-1,1)).ravel()
        true_log = target_scaler.inverse_transform(yf_val.reshape(-1,1)).ravel()
        rmse = np.sqrt(mean_squared_error(true_log, pred_log))
        fold_rmses.append(rmse)
        K.clear_session()

    mean_rmse = np.mean(fold_rmses)
    std_rmse  = np.std(fold_rmses)
    search_results.append({**params, "CV_RMSE_mean": mean_rmse, "CV_RMSE_std": std_rmse})
    print(f"  units={params['units']:3d}  drop={params['dropout']:.1f}  "
          f"lr={params['lr']:.0e}  → CV RMSE={mean_rmse:.4f} ± {std_rmse:.4f}")

    if mean_rmse < best_val_rmse:
        best_val_rmse = mean_rmse
        best_params   = params

search_df = pd.DataFrame(search_results)
print(f"\nBest params : {best_params}")
print(f"Best CV RMSE: {best_val_rmse:.4f}")

# 7.  FINAL MODEL TRAINING (full training set, best hyperparams)
print("\n─"*35)
print("  FINAL MODEL TRAINING")
print("─"*35)

K.clear_session()
model = build_shallow_lstm(LOOKBACK, N_FEATURES, **best_params)
model.summary()

callbacks = [
    EarlyStopping(patience=30, restore_best_weights=True,
                  monitor="val_loss", verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=15, min_lr=1e-6,
                      monitor="val_loss", verbose=1),
    ModelCheckpoint(f"{OUT}best_lstm_model.keras",
                    save_best_only=True, monitor="val_loss"),
]

# Use last 15% of training as validation (no data leakage)
val_split = 0.15
history = model.fit(
    X_tr, y_tr,
    epochs          = 500,
    batch_size      = 16,
    validation_split= val_split,
    callbacks       = callbacks,
    shuffle         = False,   # time series — no shuffle
    verbose         = 1,
)

print(f"\nTraining stopped at epoch {len(history.history['loss'])}")
print(f"Best val_loss : {min(history.history['val_loss']):.6f}")

# 8.  PREDICTIONS  (log-scale → original scale)
def inv_transform_pred(pred_sc):
    """Inverse MinMaxScaler → inverse log1p → original case count."""
    log_vals = target_scaler.inverse_transform(
                   pred_sc.reshape(-1, 1)).ravel()
    return np.expm1(log_vals)

lstm_train_pred_sc = model.predict(X_tr, verbose=0).ravel()
lstm_test_pred_sc  = model.predict(X_te, verbose=0).ravel()

lstm_train_pred = inv_transform_pred(lstm_train_pred_sc)
lstm_test_pred  = inv_transform_pred(lstm_test_pred_sc)

# True values in original scale
y_train_log  = target_scaler.inverse_transform(y_tr.reshape(-1,1)).ravel()
y_test_log   = target_scaler.inverse_transform(y_te.reshape(-1,1)).ravel()
y_train_orig = np.expm1(y_train_log)
y_test_orig  = np.expm1(y_test_log)

# 9.  METRICS
def compute_metrics(y_true, y_pred, label):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true+1, y_pred+1) * 100
    corr, pval = pearsonr(y_true, y_pred)
    nse  = 1 - np.sum((y_true-y_pred)**2) / np.sum((y_true-y_true.mean())**2)
    return {"Model":label,"RMSE":rmse,"MAE":mae,"R²":r2,
            "NSE":nse,"MAPE(%)":mape,"Pearson r":corr,"p-value":pval}

metrics_rows = [
    compute_metrics(y_train_orig, lstm_train_pred, "Shallow LSTM — Train"),
    compute_metrics(y_test_orig,  lstm_test_pred,  "Shallow LSTM — Test (2025)"),
]
metrics_df = pd.DataFrame(metrics_rows)

print("\n" + "─"*70)
print("  PERFORMANCE METRICS")
print("─"*70)
print(metrics_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
metrics_df.to_csv(f"{OUT}lstm_metrics.csv", index=False)
search_df.to_csv(f"{OUT}lstm_hyperparameter_search.csv", index=False)

# Corresponding date arrays for sequences
train_dates_seq = df_train["DATE"].values[LOOKBACK:]
test_dates_seq  = df_test["DATE"].values

# 10. FIGURE 1 — Training History (Loss & MAE curves)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Figure 1. Shallow LSTM — Training History\n"
             "Loss (Huber) & MAE over Epochs", fontsize=13, fontweight="bold")

ep = np.arange(1, len(history.history["loss"]) + 1)

ax = axes[0]
ax.plot(ep, history.history["loss"],     color=COLORS["lstm"], lw=2,  label="Train Loss")
ax.plot(ep, history.history["val_loss"], color=COLORS["neg"],  lw=2,  label="Val Loss", linestyle="--")
ax.axvline(np.argmin(history.history["val_loss"])+1, color="black",
           lw=1.5, ls=":", label=f"Best epoch = {np.argmin(history.history['val_loss'])+1}")
ax.set_xlabel("Epoch"); ax.set_ylabel("Huber Loss")
ax.set_title("(a) Loss Curve")
ax.legend()

ax = axes[1]
ax.plot(ep, history.history["mae"],     color=COLORS["lstm"], lw=2,  label="Train MAE")
ax.plot(ep, history.history["val_mae"], color=COLORS["neg"],  lw=2,  label="Val MAE", linestyle="--")
ax.set_xlabel("Epoch"); ax.set_ylabel("MAE (log-scale)")
ax.set_title("(b) MAE Curve")
ax.legend()

plt.tight_layout()
fig.savefig(f"{OUT}LSTM_Fig1_Training_History.png")
plt.close()
print("\nFigure 1 saved → LSTM_Fig1_Training_History.png")

# 11. FIGURE 2 — Full Time-Series: Actual vs LSTM Predicted
# Full actual series
all_dates  = df["DATE"].values[LOOKBACK:]
all_actual = np.expm1(df["DENGUE_LOG"].values[LOOKBACK:])

fig, ax = plt.subplots(figsize=(16, 6))
ax.axvspan(pd.Timestamp("2008-01-01"), pd.Timestamp("2024-12-31"),
           alpha=0.05, color="steelblue")
ax.axvspan(pd.Timestamp("2025-01-01"), pd.Timestamp("2025-11-01"),
           alpha=0.08, color="gold")

ax.plot(all_dates, all_actual,
        color=COLORS["actual"], lw=1.8, label="Actual Cases", zorder=3)
ax.plot(train_dates_seq, lstm_train_pred,
        color=COLORS["lstm"], lw=1.4, linestyle="--", alpha=0.80,
        label="LSTM Predicted (Train)", zorder=2)
ax.plot(test_dates_seq, lstm_test_pred,
        color=COLORS["lstm"], lw=2.8, linestyle="-",
        marker="o", markersize=7, label="LSTM Forecast (2025)", zorder=5)
ax.plot(test_dates_seq, y_test_orig,
        color="black", lw=2.0, linestyle="-",
        marker="s", markersize=7, alpha=0.9, label="Actual 2025", zorder=4)

# Annotate 2025 predictions
months_test = [MONTH_NAMES[pd.Timestamp(d).month-1] for d in test_dates_seq]
for d, pred, actual in zip(test_dates_seq, lstm_test_pred, y_test_orig):
    ax.annotate(f"{int(pred):,}", xy=(d, pred),
                xytext=(0, 12), textcoords="offset points",
                ha="center", fontsize=8, color=COLORS["lstm"], fontweight="bold")

ax.set_xlabel("Date"); ax.set_ylabel("Dengue Cases")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"{int(x):,}"))
ax.set_title("Figure 2. Shallow LSTM — Actual vs Predicted Dengue Cases\n"
             "Bangladesh 2008–2024 (Training) + 2025 Jan–Oct (Forecast)",
             fontweight="bold", fontsize=13)
ax.legend(ncol=2, fontsize=10)

ax.text(pd.Timestamp("2015-01-01"), ax.get_ylim()[1]*0.88,
        "Training Period (2008–2024)",
        ha="center", fontsize=9, color="steelblue", style="italic")
ax.text(pd.Timestamp("2025-05-15"), ax.get_ylim()[1]*0.88,
        "2025 Forecast",
        ha="center", fontsize=9, color="goldenrod", style="italic")

plt.tight_layout()
fig.savefig(f"{OUT}LSTM_Fig2_TimeSeries.png")
plt.close()
print("Figure 2 saved → LSTM_Fig2_TimeSeries.png")

# 12. FIGURE 3 — 2025 Monthly Bar: Actual vs LSTM Forecast
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(months_test))
w = 0.35
bars_a = ax.bar(x - w/2, y_test_orig,   width=w, label="Actual",
                color=COLORS["actual"], alpha=0.85, edgecolor="white")
bars_p = ax.bar(x + w/2, lstm_test_pred, width=w, label="LSTM Predicted",
                color=COLORS["lstm"],   alpha=0.85, edgecolor="white")

# Error percentage labels
for j, (a, p) in enumerate(zip(y_test_orig, lstm_test_pred)):
    err_pct = abs(a - p) / (a + 1) * 100
    ax.text(j, max(a, p) + 600, f"{err_pct:.1f}%",
            ha="center", va="bottom", fontsize=9, color="dimgray")

r2_val   = r2_score(y_test_orig, lstm_test_pred)
rmse_val = np.sqrt(mean_squared_error(y_test_orig, lstm_test_pred))
mae_val  = mean_absolute_error(y_test_orig, lstm_test_pred)
corr_val,_ = pearsonr(y_test_orig, lstm_test_pred)

ax.text(0.02, 0.97,
        f"R²      = {r2_val:.3f}\nRMSE = {rmse_val:,.0f}\n"
        f"MAE   = {mae_val:,.0f}\nr       = {corr_val:.3f}",
        transform=ax.transAxes, va="top", fontsize=10,
        bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="gray", alpha=0.85))

ax.set_xticks(x); ax.set_xticklabels(months_test)
ax.set_xlabel("Month (2025)"); ax.set_ylabel("Dengue Cases")
ax.set_title("Figure 3. 2025 Monthly Forecast vs Actual Dengue Cases\n"
             "Shallow LSTM — Bangladesh", fontweight="bold", fontsize=13)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"{int(x):,}"))
ax.legend()
plt.tight_layout()
fig.savefig(f"{OUT}LSTM_Fig3_2025_Forecast_Bar.png")
plt.close()
print("Figure 3 saved → LSTM_Fig3_2025_Forecast_Bar.png")

# 13. FIGURE 4 — Scatter: Actual vs Predicted (Train + Test, 4-panel)
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
fig.suptitle("Figure 4. Scatter Plots — Shallow LSTM: Actual vs Predicted",
             fontsize=13, fontweight="bold")

for ax, (label, y_t, y_p, tag) in zip(axes, [
    ("Training Set (2008–2024)", y_train_orig, lstm_train_pred, "(a)"),
    ("2025 Forecast (Jan–Oct)",  y_test_orig,  lstm_test_pred,  "(b)"),
]):
    ax.scatter(y_t, y_p, alpha=0.55, edgecolors=COLORS["lstm"],
               facecolors="none", s=60, lw=1.3)
    lo, hi = min(y_t.min(), y_p.min()), max(y_t.max(), y_p.max())
    ax.plot([lo,hi],[lo,hi],"k--", lw=1.3, label="Perfect fit (1:1)")

    slope, intercept, r, pv, _ = stats.linregress(y_t, y_p)
    x_fit = np.linspace(lo, hi, 200)
    ax.plot(x_fit, slope*x_fit+intercept, color=COLORS["lstm"],
            lw=2, label=f"OLS: y={slope:.2f}x+{intercept:.0f}")

    r2   = r2_score(y_t, y_p)
    rmse = np.sqrt(mean_squared_error(y_t, y_p))
    nse  = 1 - np.sum((y_t-y_p)**2)/np.sum((y_t-y_t.mean())**2)
    ax.set_title(f"{tag} {label}"); ax.set_xlabel("Actual"); ax.set_ylabel("Predicted")
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"{int(x):,}"))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"{int(x):,}"))
    ax.text(0.04, 0.95,
            f"R²   = {r2:.3f}\nRMSE = {rmse:,.0f}\nNSE  = {nse:.3f}\nr    = {r:.3f}",
            transform=ax.transAxes, va="top", fontsize=10,
            bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="gray", alpha=0.85))
    ax.legend(fontsize=9)

plt.tight_layout()
fig.savefig(f"{OUT}LSTM_Fig4_Scatter.png")
plt.close()
print("Figure 4 saved → LSTM_Fig4_Scatter.png")

# 14. FIGURE 5 — Residual Diagnostics (4-panel)
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle("Figure 5. Residual Diagnostics — Shallow LSTM",
             fontsize=13, fontweight="bold")

res_train = y_train_orig - lstm_train_pred
res_test  = y_test_orig  - lstm_test_pred

# (a) Residuals vs Fitted (train)
ax = axes[0, 0]
ax.scatter(lstm_train_pred, res_train, alpha=0.45, color=COLORS["lstm"],
           s=30, edgecolors="none")
ax.axhline(0, color="black", lw=1.3, ls="--")
z = np.polyfit(lstm_train_pred, res_train, 1)
xf = np.linspace(lstm_train_pred.min(), lstm_train_pred.max(), 200)
ax.plot(xf, np.polyval(z, xf), color=COLORS["neg"], lw=1.5, ls="--",
        label="Trend line")
ax.set_xlabel("Fitted Values"); ax.set_ylabel("Residuals")
ax.set_title("(a) Residuals vs Fitted (Train)")
ax.legend(fontsize=9)

# (b) Q-Q plot
ax = axes[0, 1]
(osm, osr), (slope, intercept, r) = stats.probplot(res_train, dist="norm")
ax.plot(osm, osr, "o", color=COLORS["lstm"], alpha=0.5, markersize=4)
ax.plot([osm[0], osm[-1]],
        [slope*osm[0]+intercept, slope*osm[-1]+intercept],
        "k-", lw=2)
# Shapiro-Wilk
sw_stat, sw_p = stats.shapiro(res_train[:50])   # SW works best n<50; subsample
ax.set_xlabel("Theoretical Quantiles"); ax.set_ylabel("Sample Quantiles")
ax.set_title("(b) Normal Q-Q Plot (Train Residuals)")
ax.text(0.04, 0.95, f"Shapiro-Wilk p = {sw_p:.3f}",
        transform=ax.transAxes, va="top", fontsize=9,
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray"))

# (c) Residual histogram
ax = axes[1, 0]
ax.hist(res_train, bins=30, color=COLORS["lstm"], alpha=0.7,
        edgecolor="white", density=True)
mu, sigma = res_train.mean(), res_train.std()
xn = np.linspace(res_train.min(), res_train.max(), 200)
ax.plot(xn, stats.norm.pdf(xn, mu, sigma), "k-", lw=2,
        label=f"N(μ={mu:.0f}, σ={sigma:.0f})")
ax.set_xlabel("Residuals"); ax.set_ylabel("Density")
ax.set_title("(c) Residual Distribution (Train)")
ax.legend(fontsize=9)

# (d) Test residuals per month (2025)
ax = axes[1, 1]
colors_bar = [COLORS["pos"] if r >= 0 else COLORS["neg"] for r in res_test]
bars = ax.bar(months_test, res_test, color=colors_bar, edgecolor="white")
ax.axhline(0, color="black", lw=1.3)
ax.set_xlabel("Month (2025)"); ax.set_ylabel("Residual (Actual − Predicted)")
ax.set_title("(d) Forecast Residuals — 2025 Jan–Oct")
for bar, val in zip(bars, res_test):
    ax.text(bar.get_x() + bar.get_width()/2,
            val + (600 if val >= 0 else -1200),
            f"{int(val):,}", ha="center", fontsize=8.5)

plt.tight_layout()
fig.savefig(f"{OUT}LSTM_Fig5_Residuals.png")
plt.close()
print("Figure 5 saved → LSTM_Fig5_Residuals.png")

# 15. FIGURE 6 — Hyperparameter Search Results
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Figure 6. Hyperparameter Search — Walk-Forward Cross-Validation",
             fontsize=13, fontweight="bold")

ax = axes[0]
config_labels = [f"u={r['units']}\ndo={r['dropout']}\nlr={r['lr']:.0e}"
                 for _, r in search_df.iterrows()]
bar_colors = [COLORS["pos"] if i == search_df["CV_RMSE_mean"].idxmin()
              else COLORS["lstm"] for i in range(len(search_df))]
ax.bar(range(len(search_df)), search_df["CV_RMSE_mean"],
       yerr=search_df["CV_RMSE_std"], color=bar_colors, alpha=0.85,
       edgecolor="white", capsize=5)
ax.set_xticks(range(len(search_df)))
ax.set_xticklabels(config_labels, fontsize=9)
ax.set_xlabel("Configuration"); ax.set_ylabel("CV RMSE (log-scale)")
ax.set_title("(a) Mean CV RMSE per Hyperparameter Set")
best_patch = mpatches.Patch(color=COLORS["pos"], label="Best config")
ax.legend(handles=[best_patch])

import matplotlib.patches as mpatches

# Heat-map of units × dropout
ax = axes[1]
pivot = search_df.pivot_table(index="units", columns="dropout",
                               values="CV_RMSE_mean", aggfunc="min")
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="RdYlGn_r",
            ax=ax, cbar_kws={"label":"CV RMSE (log)"})
ax.set_title("(b) CV RMSE — Units × Dropout (best lr)")
ax.set_xlabel("Dropout Rate"); ax.set_ylabel("LSTM Units")

plt.tight_layout()
fig.savefig(f"{OUT}LSTM_Fig6_HyperSearch.png")
plt.close()
print("Figure 6 saved → LSTM_Fig6_HyperSearch.png")

# 16. FIGURE 7 — Walk-Forward (TimeSeriesSplit) Validation per Fold
print("\n  Running 5-Fold Walk-Forward Validation for Figure 7 …")
K.clear_session()

tscv5        = TimeSeriesSplit(n_splits=5)
fold_metrics = []
fig, axes    = plt.subplots(5, 1, figsize=(14, 18), sharex=False)
fig.suptitle("Figure 7. Walk-Forward Validation — Shallow LSTM (5 Folds)\n"
             "Training set only (2008–2024)", fontsize=13, fontweight="bold")

for fold, (tr_idx, val_idx) in enumerate(tscv5.split(X_tr)):
    K.clear_session()
    Xf_tr, yf_tr   = X_tr[tr_idx], y_tr[tr_idx]
    Xf_val, yf_val = X_tr[val_idx], y_tr[val_idx]

    m = build_shallow_lstm(LOOKBACK, N_FEATURES, **best_params)
    m.fit(Xf_tr, yf_tr, epochs=300, batch_size=16,
          validation_data=(Xf_val, yf_val),
          callbacks=[EarlyStopping(patience=25, restore_best_weights=True)],
          verbose=0)

    pred_sc  = m.predict(Xf_val, verbose=0).ravel()
    pred_log = target_scaler.inverse_transform(pred_sc.reshape(-1,1)).ravel()
    true_log = target_scaler.inverse_transform(yf_val.reshape(-1,1)).ravel()
    pred_orig = np.expm1(pred_log)
    true_orig = np.expm1(true_log)

    r2   = r2_score(true_orig, pred_orig)
    rmse = np.sqrt(mean_squared_error(true_orig, pred_orig))
    mae  = mean_absolute_error(true_orig, pred_orig)
    fold_metrics.append({"Fold":fold+1,"R²":r2,"RMSE":rmse,"MAE":mae})

    ax = axes[fold]
    ax.plot(true_orig, color=COLORS["actual"], lw=1.8, label="Actual")
    ax.plot(pred_orig, color=COLORS["lstm"],   lw=1.8, ls="--", label="Predicted")
    ax.set_title(f"Fold {fold+1}  |  R²={r2:.3f}  RMSE={rmse:,.0f}  MAE={mae:,.0f}",
                 fontsize=11)
    ax.set_ylabel("Cases")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"{int(x):,}"))
    ax.legend(fontsize=9)

axes[-1].set_xlabel("Validation Sample Index")
plt.tight_layout()
fig.savefig(f"{OUT}LSTM_Fig7_WalkForward_Folds.png")
plt.close()
K.clear_session()
print("✔  Figure 7 saved → LSTM_Fig7_WalkForward_Folds.png")

fold_df = pd.DataFrame(fold_metrics)
print("\n  Walk-Forward Fold Summary:")
print(fold_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"\n  Mean  R² = {fold_df['R²'].mean():.4f} ± {fold_df['R²'].std():.4f}")
print(f"  Mean RMSE= {fold_df['RMSE'].mean():.1f} ± {fold_df['RMSE'].std():.1f}")
fold_df.to_csv(f"{OUT}lstm_walkforward_folds.csv", index=False)

# 17. FIGURE 8 — Annual Aggregation: Actual vs LSTM Predicted
# Re-attach training predictions to df_train (offset by LOOKBACK)
df_train_seq = df_train.iloc[LOOKBACK:].copy()
df_train_seq["LSTM_Pred"] = lstm_train_pred

annual_train = df_train_seq.groupby("YEAR").agg(
    Actual=("DENGUE","sum"), LSTM=("LSTM_Pred","sum")).reset_index()
annual_test  = pd.DataFrame({
    "YEAR":[2025], "Actual":[y_test_orig.sum()], "LSTM":[lstm_test_pred.sum()]
})
annual_all = pd.concat([annual_train, annual_test], ignore_index=True)

fig, ax = plt.subplots(figsize=(13, 6))
xr = np.arange(len(annual_all))
w  = 0.35
ax.bar(xr - w/2, annual_all["Actual"], w, label="Actual",
       color=COLORS["actual"], alpha=0.85, edgecolor="white")
ax.bar(xr + w/2, annual_all["LSTM"],   w, label="LSTM Predicted",
       color=COLORS["lstm"],   alpha=0.85, edgecolor="white")
ax.set_xticks(xr)
ax.set_xticklabels(annual_all["YEAR"].astype(int), rotation=45, ha="right")
ax.set_xlabel("Year"); ax.set_ylabel("Annual Dengue Cases")
ax.set_title("Figure 8. Annual Dengue Cases — Actual vs LSTM Predicted\n"
             "Bangladesh 2008–2025", fontweight="bold", fontsize=13)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"{int(x):,}"))
ax.axvline(len(annual_all)-1 - 0.5, color="red", ls="--", lw=1.8,
           label="→ 2025 Forecast")
ax.legend()
plt.tight_layout()
fig.savefig(f"{OUT}LSTM_Fig8_Annual.png")
plt.close()
print("Figure 8 saved → LSTM_Fig8_Annual.png")

# 18. FIGURE 9 — LSTM Architecture Diagram (text-based, publishable)
fig, ax = plt.subplots(figsize=(10, 7))
ax.axis("off")
ax.set_xlim(0, 10); ax.set_ylim(0, 10)
ax.set_facecolor("white")

# Architecture boxes
arch = [
    (5, 9.0, f"INPUT  ({LOOKBACK} × {N_FEATURES})",       "#AED6F1", 3.5, 0.55),
    (5, 7.5, f"SHALLOW LSTM  ({best_params['units']} units, tanh/sigmoid)", "#8E44AD", 5.0, 0.55),
    (5, 6.2, f"DROPOUT  (rate = {best_params['dropout']})",               "#D7BDE2", 3.5, 0.45),
    (5, 5.1, "BATCH NORMALIZATION",                        "#D7BDE2", 3.5, 0.45),
    (5, 3.9, "DENSE  (32 units, ReLU, L2)",               "#F9E79F", 3.5, 0.45),
    (5, 2.8, "OUTPUT  (1 unit, Linear)",                   "#ABEBC6", 3.5, 0.45),
]

for (x, y, txt, fc, w, h) in arch:
    bbox = dict(boxstyle="round,pad=0.4", fc=fc, ec="gray", lw=1.5)
    ax.text(x, y, txt, ha="center", va="center", fontsize=11,
            fontweight="bold", bbox=bbox,
            transform=ax.transData)

# Arrows
for i in range(len(arch)-1):
    y1 = arch[i][1]  - arch[i][5]/2
    y2 = arch[i+1][1]+ arch[i+1][5]/2
    ax.annotate("", xy=(5, y2), xytext=(5, y1),
                arrowprops=dict(arrowstyle="->", lw=2, color="#555555"))

ax.set_title("Figure 9. Shallow LSTM Architecture\n"
             f"Optimizer: Adam (lr={best_params['lr']:.0e})  |  "
             f"Loss: Huber  |  L2 reg: {best_params['l2_reg']:.0e}  |  "
             f"Lookback: {LOOKBACK} months",
             fontsize=11, fontweight="bold", pad=15)

plt.tight_layout()
fig.savefig(f"{OUT}LSTM_Fig9_Architecture.png")
plt.close()
print("Figure 9 saved → LSTM_Fig9_Architecture.png")

# 19. FIGURE 10 — Comprehensive Dashboard
fig = plt.figure(figsize=(16, 12))
fig.suptitle("Figure 10. Shallow LSTM — Comprehensive Performance Dashboard\n"
             "Bangladesh Dengue Forecast (2025)", fontsize=14, fontweight="bold")

gs = gridspec.GridSpec(3, 3, figure=fig, wspace=0.38, hspace=0.52)

# Panel A — 2025 line chart
ax0 = fig.add_subplot(gs[0, :])
ax0.plot(months_test, y_test_orig,   "ko-", lw=2, markersize=7, label="Actual 2025")
ax0.plot(months_test, lstm_test_pred,"o--", lw=2, markersize=7,
         color=COLORS["lstm"], label="LSTM Predicted")
ax0.fill_between(range(len(months_test)),
                 lstm_test_pred * 0.85, lstm_test_pred * 1.15,
                 alpha=0.15, color=COLORS["lstm"], label="±15% band")
ax0.set_title("(a) 2025 Monthly Forecast vs Actual", fontsize=11)
ax0.set_ylabel("Dengue Cases")
ax0.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"{int(x):,}"))
ax0.set_xticks(range(len(months_test))); ax0.set_xticklabels(months_test)
ax0.legend(fontsize=9, ncol=3)

# Panel B — Walk-forward fold R²
ax1 = fig.add_subplot(gs[1, 0])
ax1.bar(fold_df["Fold"], fold_df["R²"], color=COLORS["lstm"], alpha=0.8, edgecolor="white")
ax1.axhline(fold_df["R²"].mean(), color="red", ls="--", lw=1.5,
            label=f"Mean = {fold_df['R²'].mean():.3f}")
ax1.set_xlabel("Fold"); ax1.set_ylabel("R²"); ax1.set_title("(b) CV R² per Fold")
ax1.legend(fontsize=9)

# Panel C — Walk-forward fold RMSE
ax2 = fig.add_subplot(gs[1, 1])
ax2.bar(fold_df["Fold"], fold_df["RMSE"], color=COLORS["neg"], alpha=0.8, edgecolor="white")
ax2.axhline(fold_df["RMSE"].mean(), color="black", ls="--", lw=1.5,
            label=f"Mean = {fold_df['RMSE'].mean():.0f}")
ax2.set_xlabel("Fold"); ax2.set_ylabel("RMSE"); ax2.set_title("(c) CV RMSE per Fold")
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"{int(x):,}"))
ax2.legend(fontsize=9)

# Panel D — Metric summary table
ax3 = fig.add_subplot(gs[1, 2])
ax3.axis("off")
tbl_data = [
    ["Metric", "Train", "Test (2025)"],
    ["R²",     f"{r2_score(y_train_orig, lstm_train_pred):.3f}",
               f"{r2_score(y_test_orig, lstm_test_pred):.3f}"],
    ["RMSE",   f"{np.sqrt(mean_squared_error(y_train_orig, lstm_train_pred)):,.0f}",
               f"{np.sqrt(mean_squared_error(y_test_orig, lstm_test_pred)):,.0f}"],
    ["MAE",    f"{mean_absolute_error(y_train_orig, lstm_train_pred):,.0f}",
               f"{mean_absolute_error(y_test_orig, lstm_test_pred):,.0f}"],
    ["Pearson r", f"{pearsonr(y_train_orig, lstm_train_pred)[0]:.3f}",
                  f"{pearsonr(y_test_orig, lstm_test_pred)[0]:.3f}"],
    ["NSE",    f"{1 - np.sum((y_train_orig-lstm_train_pred)**2)/np.sum((y_train_orig-y_train_orig.mean())**2):.3f}",
               f"{1 - np.sum((y_test_orig-lstm_test_pred)**2)/np.sum((y_test_orig-y_test_orig.mean())**2):.3f}"],
]
tbl = ax3.table(cellText=tbl_data[1:], colLabels=tbl_data[0],
                loc="center", cellLoc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(10)
tbl.scale(1, 1.6)
for (r,c), cell in tbl.get_celld().items():
    if r == 0: cell.set_facecolor("#8E44AD"); cell.set_text_props(color="white", fontweight="bold")
    elif r % 2 == 0: cell.set_facecolor("#F4ECF7")
ax3.set_title("(d) Performance Summary", fontsize=11, pad=10)

# Panel E — Training loss zoomed
ax4 = fig.add_subplot(gs[2, :2])
ep = np.arange(1, len(history.history["loss"])+1)
ax4.plot(ep, history.history["loss"],     color=COLORS["lstm"], lw=2, label="Train")
ax4.plot(ep, history.history["val_loss"], color=COLORS["neg"],  lw=2, ls="--", label="Val")
ax4.axvline(np.argmin(history.history["val_loss"])+1, color="black", ls=":", lw=1.5)
ax4.set_xlabel("Epoch"); ax4.set_ylabel("Huber Loss"); ax4.set_title("(e) Learning Curve")
ax4.legend()

# Panel F — Residuals 2025
ax5 = fig.add_subplot(gs[2, 2])
colors_r = [COLORS["pos"] if r >= 0 else COLORS["neg"] for r in res_test]
ax5.bar(range(len(months_test)), res_test, color=colors_r, edgecolor="white")
ax5.axhline(0, color="black", lw=1.2)
ax5.set_xticks(range(len(months_test))); ax5.set_xticklabels(months_test, rotation=45)
ax5.set_title("(f) 2025 Forecast Residuals"); ax5.set_ylabel("Actual − Predicted")
ax5.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"{int(x):,}"))

fig.savefig(f"{OUT}LSTM_Fig10_Dashboard.png")
plt.close()
print("Figure 10 saved → LSTM_Fig10_Dashboard.png")

# 20.  SAVE PREDICTION TABLE
pred_2025 = df_test.copy()[["YEAR","MONTH","MIN","MAX","HUMIDITY","RAINFALL","DENGUE"]]
pred_2025.columns = ["Year","Month","Min_Temp","Max_Temp","Humidity","Rainfall","Actual_Cases"]
pred_2025["Month_Name"]      = pred_2025["Month"].apply(lambda m: MONTH_NAMES[m-1])
pred_2025["LSTM_Predicted"]  = lstm_test_pred.astype(int)
pred_2025["Residual"]        = pred_2025["Actual_Cases"] - pred_2025["LSTM_Predicted"]
pred_2025["AbsErr_Pct"]      = (pred_2025["Residual"].abs() /
                                 (pred_2025["Actual_Cases"]+1)*100).round(1)
pred_2025.to_csv(f"{OUT}lstm_2025_prediction_table.csv", index=False)

# 21.  FINAL CONSOLE REPORT
print("\n" + "="*70)
print("  FINAL SHALLOW LSTM SUMMARY")
print("="*70)
print(f"\n  Architecture : Input({LOOKBACK}×{N_FEATURES}) → LSTM({best_params['units']}) "
      f"→ Dropout({best_params['dropout']}) → BN → Dense(32) → Output(1)")
print(f"  Optimizer    : Adam  lr={best_params['lr']:.0e}  clipnorm=1.0")
print(f"  Loss fn      : Huber  |  L2 reg: {best_params['l2_reg']:.0e}")
print(f"  Epochs run   : {len(history.history['loss'])}  (EarlyStopping)")
print(f"  Parameters   : {model.count_params():,}")
print(f"\n  Train  R²    : {r2_score(y_train_orig, lstm_train_pred):.4f}")
print(f"  Test   R²    : {r2_score(y_test_orig,  lstm_test_pred):.4f}")
print(f"  Train  RMSE  : {np.sqrt(mean_squared_error(y_train_orig,lstm_train_pred)):,.1f}")
print(f"  Test   RMSE  : {np.sqrt(mean_squared_error(y_test_orig, lstm_test_pred)):,.1f}")
print(f"  Train  MAE   : {mean_absolute_error(y_train_orig,lstm_train_pred):,.1f}")
print(f"  Test   MAE   : {mean_absolute_error(y_test_orig, lstm_test_pred):,.1f}")
print(f"  Test Pearson : {pearsonr(y_test_orig, lstm_test_pred)[0]:.4f}")

print(f"\n{'─'*70}")
print("  2025 Monthly Forecast:")
print(f"{'─'*70}")
print(pred_2025[["Month_Name","Actual_Cases","LSTM_Predicted",
                  "Residual","AbsErr_Pct"]].to_string(index=False))

print(f"\n{'='*70}")
print(f"  ALL OUTPUTS SAVED TO: {OUT}")
print(f"    Figures : LSTM_Fig1 → LSTM_Fig10")
print(f"    Tables  : lstm_metrics.csv | lstm_2025_prediction_table.csv")
print(f"              lstm_hyperparameter_search.csv | lstm_walkforward_folds.csv")
print(f"    Model   : best_lstm_model.keras")
print(f"{'='*70}")

  SHALLOW LSTM — DENGUE PREDICTION, BANGLADESH (2008–2025)
  TensorFlow: 2.20.0

✔  Observations   : 211
✔  Features       : 15
✔  LSTM lookback  : 12 months
✔  Training       : 201 months (2008–2024)
✔  Test (forecast): 10 months (2025 Jan–Oct)

✔  LSTM Train     : X(189, 12, 15)  y(189,)
✔  LSTM Test      : X(10, 12, 15)  y(10,)

─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
  HYPERPARAMETER SEARCH (Walk-Forward CV)
───────────────────────────────────
  units= 32  drop=0.1  lr=1e-03  → CV RMSE=1.6887 ± 0.7596
  units= 64  drop=0.2  lr=1e-03  → CV RMSE=2.0978 ± 1.7181
  units= 64  drop=0.3  lr=5e-04  → CV RMSE=3.2526 ± 1.7612
  units=128  drop=0.2  lr=1e-03  → CV RMSE=3.0801 ± 1.6255
  units=128  drop=0.3  lr=5e-04  → CV RMSE=3.4178 ± 1.6369

✔  Best params : {'units': 32, 'dropout': 0.1, 'lr': 0.001, 'l2_reg': 0.0001}
✔  Best CV RMSE: 1.6887

─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
  FINAL MODEL TRAINING
──────────────────────────

Model: "Shallow_LSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 12, 15)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ shallow_lstm (LSTM)             │ (None, 32)             │         6,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_norm (BatchNormalization) │ (None, 32)             │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_hidden (Dense)            │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,361 (28.75 KB)

 Trainable params: 7,297 (28.50 KB)

 Non-trainable params: 64 (256.00 B)

Epoch 1/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 60ms/step - loss: 0.2418 - mae: 0.5661 - val_loss: 0.1135 - val_mae: 0.4232 - learning_rate: 0.0010
Epoch 2/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.1569 - mae: 0.4137 - val_loss: 0.1151 - val_mae: 0.4301 - learning_rate: 0.0010
Epoch 3/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.1220 - mae: 0.3711 - val_loss: 0.1451 - val_mae: 0.4946 - learning_rate: 0.0010
Epoch 4/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.1157 - mae: 0.3460 - val_loss: 0.1855 - val_mae: 0.5685 - learning_rate: 0.0010
Epoch 5/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0780 - mae: 0.2874 - val_loss: 0.2025 - val_mae: 0.5977 - learning_rate: 0.0010
Epoch 6/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0754 - mae: 0.2745 - val_loss: 0.1941 - val_mae: 0.5866 - learning_rate: 0.0010
Epoch 7/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0723 - mae: 0.2680 - val_loss: 0.1891 - val_mae: 0.5775 - learning_rate: 0.0010
Epoch 